In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from transformers import pipeline
import re
import string
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

In [ ]:
data_fake = pd.read_csv("Fake.csv")
data_true = pd.read_csv("True.csv")

In [ ]:
data_fake.head()

In [ ]:
data_true.head()

In [ ]:
data_fake["class"]=0
data_true["class"]=1


In [ ]:
data_fake.shape,data_true.shape

In [ ]:
data_fake_manual_testing = data_fake.tail(10)
for i in range(23480,23470,-1):
    data_fake.drop([i],axis=0,inplace=True)

data_true_manual_testing = data_true.tail(10)
for i in range(21416,21406,-1):
    data_true.drop([i],axis=0,inplace=True)

In [ ]:
data_fake.shape,data_true.shape

In [ ]:
data_fake_manual_testing["class"]=0
data_true_manual_testing["class"]=1

In [ ]:
data_fake_manual_testing.head()

In [ ]:
data_true_manual_testing.head()

In [ ]:
data_merge = pd.concat([data_fake,data_true],axis=0)
data_merge.head(10)

In [ ]:
data_merge.columns

In [ ]:
data=data_merge.drop(["title","subject","date"],axis=1)

In [ ]:
data.isnull().sum()

In [ ]:
data=data.sample(frac=1)

In [ ]:
data.head()

In [ ]:
data.reset_index(inplace=True)
data.drop(["index"],axis=1,inplace=True)

In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
def wordopt(text):
    text=text.lower()
    text=re.sub(r'\[.*?\]','',text)
    text=re.sub(r'\W',' ',text)
    text=re.sub(r'https?://\S+|www\.\S+','',text)
    text=re.sub(r'<.*?>+','',text)
    text=re.sub(r'[%s]'%re.escape(string.punctuation),'',text)
    text=re.sub(r'\n','',text)
    text=re.sub(r'\w*\d\w*','',text)
    return text

In [ ]:
data['text']=data['text'].apply(wordopt)

In [ ]:
x=data['text']
y=data['class']

In [ ]:
emotion_classifier = pipeline("text-classification", model="bhadresh-savani/distilbert-base-uncased-emotion", return_all_scores=True)


In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.25)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorization=TfidfVectorizer()
xv_train=vectorization.fit_transform(x_train)
xv_test=vectorization.transform(x_test)

In [ ]:
from sklearn.model_selection import cross_val_score

def evaluate_model_cv(model, name):
    print(f"\n📊 Cross-Validation for {name}:")
    scores = cross_val_score(model, xv_train, y_train, cv=5, scoring='accuracy')
    print(f"Accuracy Scores: {np.round(scores, 4)}")
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")

# Evaluate each model using cross-validation
evaluate_model_cv(LogisticRegression(), "Logistic Regression")
evaluate_model_cv(DecisionTreeClassifier(), "Decision Tree")
evaluate_model_cv(GradientBoostingClassifier(random_state=0), "Gradient Boosting")
evaluate_model_cv(RandomForestClassifier(random_state=0), "Random Forest")


In [ ]:
from sklearn.linear_model import LogisticRegression
LR=LogisticRegression()
LR.fit(xv_train,y_train)

In [ ]:
pred_lr=LR.predict(xv_test)

In [ ]:
LR.score(xv_test,y_test)

In [ ]:
print(classification_report(y_test,pred_lr))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
DT=DecisionTreeClassifier()
DT.fit(xv_train,y_train)

In [ ]:
pred_dt=DT.predict(xv_test)

In [ ]:
DT.score(xv_test,y_test)

In [ ]:
print(classification_report(y_test,pred_dt))

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
GB=GradientBoostingClassifier(random_state=0)
GB.fit(xv_train,y_train)

In [ ]:
pred_gb=GB.predict(xv_test)

In [ ]:
GB.score(xv_test,y_test)

In [ ]:
print(classification_report(y_test,pred_gb))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
RF=RandomForestClassifier(random_state=0)
RF.fit(xv_train,y_train)

In [ ]:
pred_rf=RF.predict(xv_test)

In [ ]:
RF.score(xv_test,y_test)

In [ ]:
print(classification_report(y_test,pred_rf))

In [ ]:
def get_emotion_scores(text):
    results = emotion_classifier(text[:512])  # Limit input to avoid truncation

    # Expected format: results = [[{'label': 'emotion', 'score': float}, ...]]
    # Validate the structure of results to prevent TypeError
    if not (isinstance(results, list) and
            len(results) > 0 and
            isinstance(results[0], list) and
            len(results[0]) > 0 and
            all(isinstance(item, dict) and 'score' in item and 'label' in item for item in results[0])):
        print("Warning: Unexpected output format from emotion_classifier. Returning default emotion scores.")
        return {
            "top_3_emotions": {},
            "manipulation_components": {"fear": 0, "anger": 0, "sadness": 0}
        }

    emotions = sorted(results[0], key=lambda x: x['score'], reverse=True)
    top_3 = emotions[:3]

    # Extract emotion values
    emotion_dict = {e['label']: e['score'] for e in top_3}

    # For manipulation score (Fear, Anger, Sadness)
    fear = next((e['score'] for e in results[0] if e['label'] == 'fear'), 0)
    anger = next((e['score'] for e in results[0] if e['label'] == 'anger'), 0)
    sadness = next((e['score'] for e in results[0] if e['label'] == 'sadness'), 0)

    return {
        "top_3_emotions": emotion_dict,
        "manipulation_components": {"fear": fear, "anger": anger, "sadness": sadness}
    }

In [ ]:
def output_label(n):
    if n == 0:
        return "Fake News"
    elif n == 1:
        return "Not Fake News"

In [ ]:
import math

def manual_testing(news):
    testing_news = {"text": [news]}
    new_def_test = pd.DataFrame(testing_news)
    new_def_test["text"] = new_def_test["text"].apply(wordopt)
    new_xv_test = vectorization.transform(new_def_test["text"])

    pred_LR = LR.predict(new_xv_test)
    pred_DT = DT.predict(new_xv_test)
    pred_GB = GB.predict(new_xv_test)
    pred_RF = RF.predict(new_xv_test)

    emotion_results = get_emotion_scores(news)

    fear = emotion_results["manipulation_components"]["fear"]
    anger = emotion_results["manipulation_components"]["anger"]
    sadness = emotion_results["manipulation_components"]["sadness"]

    # Advanced manipulation formula (non-linear, weighted)
    def compute_manipulation_score(fear, anger, sadness):
        # Weights based on psychological studies on emotional manipulation
        weighted_sum = (0.5 * fear) + (0.3 * anger) + (0.2 * sadness)

        # Apply a sigmoid-like curve for realism (scale and shape control)
        scaled = 1 / (1 + math.exp(-10 * (weighted_sum - 0.3)))  # threshold ~0.3
        return round(scaled * 100, 2)

    manipulation_score = compute_manipulation_score(fear, anger, sadness) if pred_LR[0] == 0 else 0

    print("\n\n🧠 Emotion Breakdown (Top 3):")
    for emotion, score in emotion_results["top_3_emotions"].items():
        print(f"{emotion}: {round(score * 100, 2)}%")

    print("\n⚠️ Manipulation Score:", manipulation_score)

    print("\n🔍 Predictions:")
    print("LR Prediction:", output_label(pred_LR[0]))
    print("DT Prediction:", output_label(pred_DT[0]))
    print("GB Prediction:", output_label(pred_GB[0]))
    print("RF Prediction:", output_label(pred_RF[0]))


In [ ]:
new = input("Enter the news: ")
manual_testing(new)

In [ ]:
import joblib

# Save TF-IDF vectorizer
joblib.dump(vectorization, "vectorizer.pkl")

# Save all models
joblib.dump(LR, "logistic_model.pkl")
joblib.dump(DT, "decision_tree.pkl")
joblib.dump(GB, "gradient_boosting.pkl")
joblib.dump(RF, "random_forest.pkl")


In [ ]:
from google.colab import files
files.download("vectorizer.pkl")
files.download("logistic_model.pkl")
files.download("decision_tree.pkl")
files.download("gradient_boosting.pkl")
files.download("random_forest.pkl")


In [ ]:
import pickle


In [ ]:
with open("logistic_regression.pkl", "wb") as f:
    pickle.dump(LR, f)


In [ ]:
with open("decision_tree.pkl", "wb") as f:
    pickle.dump(DT, f)

In [ ]:
with open("gradient_boosting.pkl", "wb") as f:
    pickle.dump(GB, f)

In [ ]:
with open("random_forest.pkl", "wb") as f:
    pickle.dump(RF, f)

In [ ]:
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorization, f)

In [ ]:
emotion_classifier = pipeline(
    "text-classification",
    model="bhadresh-savani/distilbert-base-uncased-emotion",
    return_all_scores=True
)

In [ ]:
emotion_classifier.model.save_pretrained("emotion_model")
emotion_classifier.tokenizer.save_pretrained("emotion_model")

In [ ]:
from google.colab import files

files.download("logistic_regression.pkl")
files.download("decision_tree.pkl")
files.download("gradient_boosting.pkl")
files.download("random_forest.pkl")
files.download("tfidf_vectorizer.pkl")

In [ ]:
!zip -r emotion_model.zip emotion_model

In [ ]:
files.download("emotion_model.zip")